EXPERIMENT CONFIGURATION — EDIT ONLY THIS CELL

In [ ]:
import re
import subprocess
from pathlib import Path
import shutil, time, json, requests
from datetime import datetime


In [ ]:
# -------------------------
# Model & Triton settings
# -------------------------
MODEL_NAME = "food_classifier_onnx"
TRITON_HTTP = "http://triton:8000"

# Triton model repository (mounted into both Triton + Jupyter)
MODEL_REPO = Path("/models")
ACTIVE_CONFIG = MODEL_REPO / MODEL_NAME / "config.pbtxt"

# Config template to activate for this run
CONFIG_TEMPLATE = Path(
    "/workspace/configs/food_classifier_onnx/poisson_no_batch/config.pbtxt"
)

# Must match docker-compose
REPOSITORY_POLL_SECS = 2


# Request Rate settings (requests per second range)
RPS_START = 100
RPS_END   = 500
RPS_STEP  = 50

REQUEST_DIST = "poisson"
BATCH = 1 # how many requests the client sends at once, keep to 1

# -------------------------
# Output & bookkeeping
# -------------------------
RESULTS_ROOT = Path("/workspace/results")
TIMESTAMP = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

RUN_NAME = f"{MODEL_NAME}_poisson_no_batch_{RPS_START}-{RPS_END}rps_{TIMESTAMP}"
RUN_DIR = RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# ACTIVATE CONFIG (POLL MODE)
# ============================================================

if not CONFIG_TEMPLATE.exists():
    raise FileNotFoundError(f"Config template not found: {CONFIG_TEMPLATE}")

if not ACTIVE_CONFIG.parent.exists():
    raise FileNotFoundError(f"Model directory not found: {ACTIVE_CONFIG.parent}")

shutil.copyfile(CONFIG_TEMPLATE, ACTIVE_CONFIG)
print(f"Activated config:\n  {CONFIG_TEMPLATE}\n→ {ACTIVE_CONFIG}")

# ============================================================
# WAIT FOR TRITON TO RELOAD MODEL
# ============================================================

def wait_ready(url, timeout_s=90):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            r = requests.get(url, timeout=2)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.5)
    raise TimeoutError(f"Timed out waiting for {url}")

# Give poll loop time to detect filesystem change
time.sleep(REPOSITORY_POLL_SECS + 0.5)

wait_ready(f"{TRITON_HTTP}/v2/health/ready", timeout_s=60)
wait_ready(f"{TRITON_HTTP}/v2/models/{MODEL_NAME}/ready", timeout_s=90)

print("Triton and model ready after config reload.")

# ============================================================
# RECORD EFFECTIVE MODEL CONFIG (GROUND TRUTH)
# ============================================================

effective_config = requests.get(
    f"{TRITON_HTTP}/v2/models/{MODEL_NAME}/config", timeout=5
).json()

(RUN_DIR / "effective_model_config.json").write_text(
    json.dumps(effective_config, indent=2)
)

# ============================================================
# RECORD EXPERIMENT METADATA
# ============================================================

experiment_metadata = {
    "model_name": MODEL_NAME,
    "config_template": str(CONFIG_TEMPLATE),
    "active_config": str(ACTIVE_CONFIG),
    "client_settings": {
        "request_distribution": REQUEST_DIST,
        "batch_size": BATCH,
        "rps_start": RPS_START,
        "rps_end": RPS_END,
        "rps_step": RPS_STEP,
    },
    "repository_poll_secs": REPOSITORY_POLL_SECS,
    "timestamp_utc": TIMESTAMP,
}

(RUN_DIR / "experiment.json").write_text(
    json.dumps(experiment_metadata, indent=2)
)

print(f"Experiment initialized. Results will be written to:\n  {RUN_DIR}")


Set up results directory

In [ ]:
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# copy the config used for these results
shutil.copyfile(CONFIG_TEMPLATE, RUN_DIR / "config_used.pbtxt")

Run perf analyzer benchmarks

In [ ]:
# ---- perf_analyzer fixed settings ) ----
TRITON_PERF_URL = "triton:8000"  
INPUT_NAME = "IMAGE"
INPUT_SHAPE = "3,224,224"

#percentiles from perf_analyzer:
PERCENTILES = [50, 95, 99]

# measurement behavior 
MEASUREMENT_INTERVAL_MS = 10000  # 10s window per trial
MAX_TRIALS = 10

# -----------------------------------------------------------

def run_perf_analyzer(rps: int) -> tuple[str, str]:
    """
    Returns (stdout, stderr) for one perf_analyzer invocation.
    """
    cmd = [
        "perf_analyzer",
        "-u", TRITON_PERF_URL,
        "-m", MODEL_NAME,
        "-b", str(BATCH),
        "--shape", f"{INPUT_NAME}:{INPUT_SHAPE}",
        "--request-distribution", REQUEST_DIST,
        "--request-rate-range", str(rps),  # single point (your sweep loop handles range)
        "--measurement-interval", str(MEASUREMENT_INTERVAL_MS),
        "--max-trials", str(MAX_TRIALS),
    ]

    # Add percentile flags if you want them explicitly (supported in recent perf_analyzer)
    for p in PERCENTILES:
        cmd += ["--percentile", str(p)]

    completed = subprocess.run(
        cmd, capture_output=True, text=True, check=False
    )
    return completed.stdout, completed.stderr


In [ ]:
def parse_perf_analyzer_output(text: str) -> dict:
    """
    Parse perf_analyzer output
    Returns:
      - request_rate_rps
      - client_request_count
      - throughput_infer_per_s
      - avg_latency_ms
      - p50/p90/p95/p99 latency (ms)
      - server_avg_queue_us, server_avg_infer_us, server_avg_request_us (if present)
    """
    out = {
        "request_rate_rps": None,
        "client": {
            "request_count": None,
            "throughput_infer_per_s": None,
            "avg_latency_ms": None,
            "latency_ms": {"p50": None, "p90": None, "p95": None, "p99": None},
        },
        "server": {
            "inference_count": None,
            "execution_count": None,
            "successful_request_count": None,
            "avg_request_latency_ms": None,
            "avg_queue_ms": None,
            "avg_compute_infer_ms": None,
        },
        "parse_notes": [],
    }

    # Request Rate line: 
    m = re.search(r"Request Rate:\s*([0-9.]+)\s+inference requests per second", text)
    if m:
        out["request_rate_rps"] = float(m.group(1))

    # Client request count
    m = re.search(r"Client:\s*[\s\S]*?Request count:\s*([0-9]+)", text)
    if m:
        out["client"]["request_count"] = int(m.group(1))

    # Throughput: 
    m = re.search(r"Throughput:\s*([0-9.]+)\s*infer/sec", text)
    if m:
        out["client"]["throughput_infer_per_s"] = float(m.group(1))

    # Avg latency: 
    m = re.search(r"Avg latency:\s*([0-9.]+)\s*usec", text)
    if m:
        out["client"]["avg_latency_ms"] = float(m.group(1)) / 1000.0

    # Percentiles: 
    for p in ["50", "90", "95", "99"]:
        mp = re.search(rf"p{p}\s+latency:\s*([0-9.]+)\s*usec", text)
        if mp:
            out["client"]["latency_ms"][f"p{p}"] = float(mp.group(1)) / 1000.0

    # Server counts
    m = re.search(r"Server:\s*[\s\S]*?Inference count:\s*([0-9]+)", text)
    if m:
        out["server"]["inference_count"] = int(m.group(1))
    m = re.search(r"Execution count:\s*([0-9]+)", text)
    if m:
        out["server"]["execution_count"] = int(m.group(1))
    m = re.search(r"Successful request count:\s*([0-9]+)", text)
    if m:
        out["server"]["successful_request_count"] = int(m.group(1))

    # Avg request latency breakdown:
    # "Avg request latency: 1428 usec (... + queue 128 usec + ... + compute infer 1147 usec ...)"
    m = re.search(r"Avg request latency:\s*([0-9.]+)\s*usec", text)
    if m:
        out["server"]["avg_request_latency_ms"] = float(m.group(1)) / 1000.0

    m = re.search(r"\bqueue\s*([0-9.]+)\s*usec", text)
    if m:
        out["server"]["avg_queue_ms"] = float(m.group(1)) / 1000.0

    m = re.search(r"compute infer\s*([0-9.]+)\s*usec", text)
    if m:
        out["server"]["avg_compute_infer_ms"] = float(m.group(1)) / 1000.0

    # If key fields missing, note it
    if out["client"]["throughput_infer_per_s"] is None:
        out["parse_notes"].append("Missing client throughput.")
    if out["client"]["latency_ms"]["p95"] is None:
        out["parse_notes"].append("Missing p95 latency.")
    if out["server"]["avg_queue_ms"] is None:
        out["parse_notes"].append("Missing server queue latency breakdown.")

    return out


In [ ]:
results = []

raw_dir = RUN_DIR / "perf_raw"
raw_dir.mkdir(parents=True, exist_ok=True)

for rps in range(RPS_START, RPS_END + 1, RPS_STEP):
    print(f"Running perf_analyzer at {rps} rps...")

    stdout, stderr = run_perf_analyzer(rps)

    # Save raw output (debug + provenance)
    (raw_dir / f"rps_{rps}.stdout.txt").write_text(stdout)
    (raw_dir / f"rps_{rps}.stderr.txt").write_text(stderr)

    parsed = parse_perf_analyzer_output(stdout)

    point = {
        "rps_target": rps,        
        **parsed,        
        "stderr_nonempty": bool(stderr.strip()),
    }

    results.append(point)

run_record = {
    "experiment": experiment_metadata,
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "effective_model_config": str(RUN_DIR / "effective_model_config.json"),
    "points": results,
}

(RUN_DIR / "run.json").write_text(
    json.dumps(run_record, indent=2)
)

print(f"Saved structured results to {RUN_DIR / 'run.json'}")
